# Notebook 02 - Classification Model Comparison

This notebook revises important classification algorithms and compares them manually.

Algorithms covered:

1. Logistic Regression
2. Decision Tree
3. Random Forest
4. XGBoost

**MLflow connection:** Each model run will later become an MLflow run with parameters, metrics, and artifacts.

In [ ]:
#!pip3 install xgboost

In [ ]:
# Core libraries used throughout the Day 1 notebooks
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer



from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

# XGBoost is optional in some environments. We handle import gracefully.
try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception as e:
    XGBOOST_AVAILABLE = False
    print('XGBoost is not available. Install it using: pip install xgboost')
    print('Error:', e)

In [ ]:
churn_df = pd.read_csv('../datasets/customer_churn_training.csv')
X = churn_df.drop(columns=['customer_id', 'churn'])
y = churn_df['churn']

numerical_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipeline, numerical_features),
    ('cat', categorical_pipeline, categorical_features)
])

## Helper function for evaluation

This function calculates common classification metrics.

Later in MLflow, these values will be logged using `mlflow.log_metric()`.

In [ ]:
def evaluate_classification_model(model_name, model, X_test, y_test):
    """
    Evaluate a classification model using common metrics.

    Parameters
    ----------
    model_name : str
        Name of the algorithm/model.
    model : sklearn Pipeline
        Trained model pipeline.
    X_test : DataFrame
        Test features.
    y_test : Series
        Actual target values.

    Returns
    -------
    dict
        Dictionary containing model name and evaluation metrics.
    """
    y_pred = model.predict(X_test)

    # Some models provide probability scores.
    # ROC-AUC needs probability scores for the positive class.
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_test)[:, 1]
        roc_auc = roc_auc_score(y_test, y_proba)
    else:
        roc_auc = np.nan

    metrics = {
        'model_name': model_name,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1_score': f1_score(y_test, y_pred, zero_division=0),
        'roc_auc': roc_auc
    }

    print('==============================')
    print(model_name)
    print('==============================')
    print('Confusion Matrix:')
    print(confusion_matrix(y_test, y_pred))
    print('Classification Report:')
    print(classification_report(y_test, y_pred, zero_division=0))

    return metrics

## 1. Logistic Regression

Layman idea: Logistic Regression draws a decision boundary and predicts probability of class 0 or class 1.

Useful parameters:

- `C`: regularization strength
- `max_iter`: maximum training iterations
- `solver`: optimization algorithm

In [ ]:
logistic_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(C=1.0, max_iter=1000, solver='liblinear'))
])

logistic_pipeline.fit(X_train, y_train)
logistic_metrics = evaluate_classification_model('Logistic Regression', logistic_pipeline, X_test, y_test)

## 2. Decision Tree

Layman idea: A Decision Tree asks a sequence of yes/no questions and reaches a final decision.

Useful parameters:

- `max_depth`: controls how deep the tree can grow
- `min_samples_split`: minimum samples needed to split a node
- `criterion`: splitting quality measure

Important risk: Decision Trees can overfit if they become too deep.

In [ ]:
tree_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', DecisionTreeClassifier(max_depth=5, min_samples_split=10, random_state=42))
])

tree_pipeline.fit(X_train, y_train)
tree_metrics = evaluate_classification_model('Decision Tree', tree_pipeline, X_test, y_test)

## 3. Random Forest

Layman idea: Random Forest trains many decision trees and combines their votes.

Useful parameters:

- `n_estimators`: number of trees
- `max_depth`: depth of each tree
- `min_samples_split`: minimum samples required for splitting

Random Forest usually performs better than a single Decision Tree because it reduces overfitting.

In [ ]:
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(n_estimators=150, max_depth=8, random_state=42))
])

rf_pipeline.fit(X_train, y_train)
rf_metrics = evaluate_classification_model('Random Forest', rf_pipeline, X_test, y_test)

## 4. XGBoost

Layman idea: XGBoost builds trees one after another. Each new tree tries to correct previous mistakes.

Useful parameters:

- `n_estimators`: number of boosting rounds
- `learning_rate`: how strongly each tree contributes
- `max_depth`: tree depth
- `subsample`: percentage of rows used per tree
- `colsample_bytree`: percentage of columns used per tree

This is excellent for showing MLflow because we often run many XGBoost experiments with different parameters.

In [ ]:
if XGBOOST_AVAILABLE:
    xgb_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', XGBClassifier(
            n_estimators=120,
            learning_rate=0.08,
            max_depth=3,
            subsample=0.9,
            colsample_bytree=0.9,
            eval_metric='logloss',
            random_state=42
        ))
    ])

    xgb_pipeline.fit(X_train, y_train)
    xgb_metrics = evaluate_classification_model('XGBoost', xgb_pipeline, X_test, y_test)
else:
    xgb_metrics = {
        'model_name': 'XGBoost',
        'accuracy': np.nan,
        'precision': np.nan,
        'recall': np.nan,
        'f1_score': np.nan,
        'roc_auc': np.nan
    }
    print('Skipped XGBoost because package is not installed.')

## Compare all models manually

This table is what MLflow will help us create and maintain automatically.

In [ ]:
results_df = pd.DataFrame([
    logistic_metrics,
    tree_metrics,
    rf_metrics,
    xgb_metrics
]).sort_values(by='f1_score', ascending=False)

results_df

In [ ]:
# Save manual comparison result as an artifact-like output.
# In MLflow, this kind of file can be logged as an artifact.
results_df.to_csv('../outputs/classification_model_comparison.csv', index=False)
print('Saved comparison file to ../outputs/classification_model_comparison.csv')